# ML3. Часть 4. Cross-validation / Кросс-валидация через импорт

Этот ноутбук аналогичен `ml3_04_cross-valid.ipynb`, но сами методы кросс-валидации не реализуются внутри ноутбука. Они импортируются из файла:

`cross_validation_methods.py`

Такой вариант удобнее для дальнейшей работы: функции можно переиспользовать в разных ноутбуках без копирования кода.

## 1. Импорт библиотек, модуля с методами и загрузка данных

Файл с функциями лежит в каталоге `src`. Чтобы импорт работал и при запуске ноутбука из корня проекта, и при запуске из папки `src`, ниже автоматически определяется путь к `src` и добавляется в `sys.path`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

current_dir = Path.cwd()

if (current_dir / "src" / "cross_validation_methods.py").exists():
    src_dir = current_dir / "src"
elif (current_dir / "cross_validation_methods.py").exists():
    src_dir = current_dir
else:
    raise FileNotFoundError("Не найден файл cross_validation_methods.py")

if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

from cross_validation_methods import (
    make_group_k_fold_splits,
    make_k_fold_splits,
    make_quantile_stratification_field,
    make_stratified_k_fold_splits,
    make_time_series_splits,
)

DATA_PATH = src_dir / "data" / "train_df_processed_0.75.csv"

df = pd.read_csv(DATA_PATH)
df["created"] = pd.to_datetime(df["created"])
df["price_bin"] = make_quantile_stratification_field(
    df,
    target_field="price",
    n_bins=5,
)

print(f"Используемый модуль: {src_dir / 'cross_validation_methods.py'}")
print(f"Размер датасета: {df.shape}")
display(df.head())
display(df.dtypes)

Используемый модуль: C:\21school\ML3_Validation_ID_1254800-1\src\cross_validation_methods.py
Размер датасета: (36284, 25)


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace,price_bin
0,1.0,0,2725,0,2016-05-26 15:01:42,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,1
1,1.0,3,5200,0,2016-06-16 05:42:00,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,4
2,1.0,2,4204,1,2016-04-20 02:00:53,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,3
3,1.0,1,3500,0,2016-04-22 05:35:23,1,0,0,0,1,...,0,1,0,1,0,0,1,0,0,3
4,1.0,1,2195,0,2016-05-20 01:22:06,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


bathrooms                   float64
bedrooms                      int64
price                         int64
interest                      int64
created              datetime64[us]
Elevator                      int64
HardwoodFloors                int64
CatsAllowed                   int64
DogsAllowed                   int64
Doorman                       int64
Dishwasher                    int64
NoFee                         int64
LaundryinBuilding             int64
FitnessCenter                 int64
Pre-War                       int64
LaundryinUnit                 int64
RoofDeck                      int64
OutdoorSpace                  int64
DiningRoom                    int64
HighSpeedInternet             int64
Balcony                       int64
SwimmingPool                  int64
NewConstruction               int64
Terrace                       int64
price_bin                     int64
dtype: object

## 2. Вспомогательная функция для просмотра результата

Методы из `cross_validation_methods.py` возвращают список пар `(train_indices, test_indices)`. Индексы позиционные, то есть применяются через `.iloc`.

Функция ниже не является частью задания; она нужна только для компактного вывода размеров фолдов и проверки, что `train` и `test` внутри одного фолда не пересекаются.

In [2]:
def describe_splits(splits, title: str) -> None:
    """Печатает краткое описание списка train/test индексов."""
    print(title)
    print(f"Количество фолдов: {len(splits)}")

    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        intersection = np.intersect1d(train_idx, test_idx)
        print(
            f"Fold {fold_number}: "
            f"train={len(train_idx)}, test={len(test_idx)}, "
            f"intersection={len(intersection)}"
        )

## 3. K-Fold

`K-Fold` делит датасет на `k` примерно равных частей. На каждой итерации один фолд используется как `test`, остальные фолды объединяются в `train`.

Здесь используется импортированная функция `make_k_fold_splits`.

In [3]:
k_fold_splits = make_k_fold_splits(df, k=5)
describe_splits(k_fold_splits, "K-Fold")

train_idx, test_idx = k_fold_splits[0]
df_train_fold = df.iloc[train_idx]
df_test_fold = df.iloc[test_idx]

display(df_train_fold.head())
display(df_test_fold.head())

K-Fold
Количество фолдов: 5
Fold 1: train=29027, test=7257, intersection=0
Fold 2: train=29027, test=7257, intersection=0
Fold 3: train=29027, test=7257, intersection=0
Fold 4: train=29027, test=7257, intersection=0
Fold 5: train=29028, test=7256, intersection=0


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace,price_bin
14466,1.0,2,2660,0,2016-06-23 02:26:03,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
31828,2.0,3,3734,0,2016-05-14 03:07:51,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
8565,1.0,1,2590,1,2016-05-14 04:08:08,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,1
11540,1.0,2,3595,2,2016-06-13 02:54:34,1,1,1,1,0,...,1,0,0,0,0,1,0,0,0,3
1204,1.0,1,2295,1,2016-05-06 04:51:40,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace,price_bin
23895,2.0,4,5400,1,2016-06-21 04:26:15,1,1,1,1,1,...,0,1,0,1,0,0,0,0,0,4
19580,1.0,1,4000,0,2016-06-14 12:38:24,1,1,0,0,1,...,1,0,0,1,0,0,0,0,0,3
7124,2.0,3,4100,1,2016-06-03 05:41:15,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,3
19128,1.0,1,2885,1,2016-04-30 05:57:40,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,2
441,1.0,1,2485,0,2016-05-05 05:20:46,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,1


## 4. Grouped K-Fold

`Grouped K-Fold` нужен, когда объекты одной группы нельзя разделять между `train` и `test` внутри одного фолда.

В предобработанном датасете нет явного идентификатора клиента или здания, поэтому для демонстрации используем `bedrooms` как технический пример `group_field`. В реальной задаче лучше подставить настоящий групповой идентификатор, если он есть.

In [4]:
group_k_fold_splits = make_group_k_fold_splits(df, k=3, group_field="bedrooms")
describe_splits(group_k_fold_splits, "Grouped K-Fold по bedrooms")

for fold_number, (train_idx, test_idx) in enumerate(group_k_fold_splits, start=1):
    train_groups = set(df.iloc[train_idx]["bedrooms"])
    test_groups = set(df.iloc[test_idx]["bedrooms"])
    print(
        f"Fold {fold_number}: "
        f"train_groups={sorted(train_groups)}, "
        f"test_groups={sorted(test_groups)}, "
        f"group_intersection={sorted(train_groups & test_groups)}"
    )

Grouped K-Fold по bedrooms
Количество фолдов: 3
Fold 1: train=24447, test=11837, intersection=0
Fold 2: train=23958, test=12326, intersection=0
Fold 3: train=24163, test=12121, intersection=0
Fold 1: train_groups=[0, 2, 3, 4], test_groups=[1, 5, 6, 7, 8], group_intersection=[]
Fold 2: train_groups=[0, 1, 3, 5, 6, 7, 8], test_groups=[2, 4], group_intersection=[]
Fold 3: train_groups=[1, 2, 4, 5, 6, 7, 8], test_groups=[0, 3], group_intersection=[]


## 5. Stratified K-Fold

`Stratified K-Fold` сохраняет распределение выбранного поля в каждом фолде. Так как в этом проекте целевая переменная - `price`, задача является регрессионной.

Непрерывный `price` нельзя напрямую использовать как `stratify_field`: почти каждое значение может оказаться отдельным классом. Поэтому сначала создаем дискретное поле `price_bin`, разбивая `price` на квантили через `make_quantile_stratification_field`. Затем в `make_stratified_k_fold_splits` передаем именно `price_bin`, чтобы фолды сохраняли похожее распределение ценовых диапазонов.

In [5]:
stratified_k_fold_splits = make_stratified_k_fold_splits(
    df,
    k=5,
    stratify_field="price_bin",
)
describe_splits(stratified_k_fold_splits, "Stratified K-Fold по price_bin")

print("Распределение price_bin в полном датасете:")
display(df["price_bin"].value_counts(normalize=True).sort_index().rename("full_share"))

print("Границы и количество объектов в бинах price:")
display(
    df.groupby("price_bin")["price"]
    .agg(["min", "max", "count"])
    .rename_axis("price_bin")
)

for fold_number, (_, test_idx) in enumerate(stratified_k_fold_splits, start=1):
    print(f"Fold {fold_number}, распределение price_bin в test:")
    display(
        df.iloc[test_idx]["price_bin"]
        .value_counts(normalize=True)
        .sort_index()
        .rename("test_share")
    )
    print("Описание price в test:")
    display(df.iloc[test_idx]["price"].describe())

Stratified K-Fold по price_bin
Количество фолдов: 5
Fold 1: train=29026, test=7258, intersection=0
Fold 2: train=29026, test=7258, intersection=0
Fold 3: train=29027, test=7257, intersection=0
Fold 4: train=29028, test=7256, intersection=0
Fold 5: train=29029, test=7255, intersection=0
Распределение price_bin в полном датасете:


price_bin
0    0.200088
1    0.201824
2    0.204084
3    0.196560
4    0.197442
Name: full_share, dtype: float64

Границы и количество объектов в бинах price:


,min,max,count
price_bin,,,
0,1475,2390,7260
1,2391,2875,7323
2,2877,3450,7405
3,3451,4495,7132
4,4496,13000,7164


Fold 1, распределение price_bin в test:


...

count     7256.000000
mean      3547.940325
std       1622.670550
min       1489.000000
25%       2500.000000
50%       3150.000000
75%       4050.000000
max      12995.000000
Name: price, dtype: float64

Fold 5, распределение price_bin в test:


price_bin
0    0.200138
1    0.201792
2    0.204135
3    0.196554
4    0.197381
Name: test_share, dtype: float64

Описание price в test:


count     7255.000000
mean      3537.856237
std       1598.500673
min       1475.000000
25%       2500.000000
50%       3150.000000
75%       4050.000000
max      13000.000000
Name: price, dtype: float64

## 6. Time series split

`Time series split` используется для данных с временной зависимостью. Данные сортируются по `date_field`; модель обучается на прошлом и проверяется на следующем временном блоке.

Здесь используем поле `created`.

In [6]:
time_series_splits = make_time_series_splits(df, k=5, date_field="created")
describe_splits(time_series_splits, "Time series split по created")

for fold_number, (train_idx, test_idx) in enumerate(time_series_splits, start=1):
    train_dates = df.iloc[train_idx]["created"]
    test_dates = df.iloc[test_idx]["created"]
    print(
        f"Fold {fold_number}: "
        f"train {train_dates.min()} - {train_dates.max()}, "
        f"test {test_dates.min()} - {test_dates.max()}"
    )

Time series split по created
Количество фолдов: 4
Fold 1: train=7257, test=7257, intersection=0
Fold 2: train=14514, test=7257, intersection=0
Fold 3: train=21771, test=7257, intersection=0
Fold 4: train=29028, test=7256, intersection=0
Fold 1: train 2016-04-01 22:12:41 - 2016-04-19 03:40:22, test 2016-04-19 03:40:26 - 2016-05-07 02:52:37
Fold 2: train 2016-04-01 22:12:41 - 2016-05-07 02:52:37, test 2016-05-07 02:52:51 - 2016-05-24 14:55:17
Fold 3: train 2016-04-01 22:12:41 - 2016-05-24 14:55:17, test 2016-05-24 14:57:06 - 2016-06-12 08:29:35
Fold 4: train 2016-04-01 22:12:41 - 2016-06-12 08:29:35, test 2016-06-12 08:29:43 - 2016-06-29 21:41:47


## 7. Проверка единого формата результата

Все импортированные методы возвращают одинаковую структуру: список фолдов, где каждый фолд - это пара массивов `train_indices` и `test_indices`.

Это удобно для дальнейшего обучения моделей: один и тот же цикл можно использовать для разных схем кросс-валидации.

In [7]:
all_split_examples = {
    "K-Fold": k_fold_splits,
    "Grouped K-Fold": group_k_fold_splits,
    "Stratified K-Fold": stratified_k_fold_splits,
    "Time series split": time_series_splits,
}

for split_name, splits in all_split_examples.items():
    first_train_idx, first_test_idx = splits[0]
    print(split_name)
    print("Тип результата:", type(splits))
    print("Тип train индексов:", type(first_train_idx), "размер:", first_train_idx.shape)
    print("Тип test индексов:", type(first_test_idx), "размер:", first_test_idx.shape)
    print("Пример df_train shape:", df.iloc[first_train_idx].shape)
    print("Пример df_test shape:", df.iloc[first_test_idx].shape)
    print("-" * 60)

K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (29027,)
Тип test индексов: <class 'numpy.ndarray'> размер: (7257,)
Пример df_train shape: (29027, 25)
Пример df_test shape: (7257, 25)
------------------------------------------------------------
Grouped K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (24447,)
Тип test индексов: <class 'numpy.ndarray'> размер: (11837,)
Пример df_train shape: (24447, 25)
Пример df_test shape: (11837, 25)
------------------------------------------------------------
Stratified K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (29026,)
Тип test индексов: <class 'numpy.ndarray'> размер: (7258,)
Пример df_train shape: (29026, 25)
Пример df_test shape: (7258, 25)
------------------------------------------------------------
Time series split
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (7257,)
Тип test 

## 8. Итог

В этом ноутбуке используются методы, импортированные из `cross_validation_methods.py`:

* `make_k_fold_splits`;
* `make_quantile_stratification_field`;
* `make_group_k_fold_splits`;
* `make_stratified_k_fold_splits`;
* `make_time_series_splits`.

Такой подход лучше подходит для проекта: реализация хранится в `.py`-модуле, а ноутбук остается местом для экспериментов, проверок и пояснений. Для регрессионной цели `price` стратификация выполняется по квантильным бинам `price_bin`.